# REBULEX — LLM özet şablonu testi + mini B (Colab)

`summarize_llm.py` 5 hedef kararı (T01, T12, T21, T28, T29) `magibu/magibu-11b-v0.8` ile özetler, özetleri
Magibu embedding'iyle vektörleştirir ve bu 5 hedefin sorgularında (15 v2 + 5 dilekçe) özetsiz / özetli sırayı basar.
Kod GitHub'dan (`metin-temizleme` dalı), veriler Drive'dan gelir.

**Başlamadan önce**
1. `Çalışma zamanı > Çalışma zamanı türünü değiştir` → **A100**.
2. Drive'da `REBULEX/rerank_veri.zip` olmalı.
3. Soldaki 🔑 *Secrets* bölümünde `HF_TOKEN` olmalı (notebook erişimi açık). Model gated ("auto" onay):
   Hugging Face'te model sayfasında lisansı bir kez onaylamış olmalısın.

**Akış:** `Çalışma zamanı > Tümünü çalıştır`. 1. hücrede Drive izni (bazen 3. hücrede HF_TOKEN erişimi) istenir,
onayla. Her özet bittiği anda Drive'a (`REBULEX/ozet_sonuc/`) yazılır; kesilirse yeniden "Tümünü çalıştır",
bitmiş özetler atlanır. Son hücre çalışma zamanını kapatır.

**Uyarı:** Yalnızca hedeflerin özeti olduğu için özetli sıra hedeflere haksız avantaj verir. Bu bir boru hattı testi,
kazanç ölçümü değil.

## 1. Drive'ı bağla

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Ayarlar

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/REBULEX"
SONUC = f"{DRIVE_DIR}/ozet_sonuc"
LOCAL_DATA = "/content/data"
REPO = "https://github.com/ahmetege0/rebulex.git"
BRANCH = "metin-temizleme"

os.environ["REBULEX_DATA"] = LOCAL_DATA     # config.py veriyi buradan okur
os.environ["REBULEX_DTYPE"] = "bfloat16"    # embedding modeli için

## 3. Kodu GitHub'dan çek ve kur
Kütüphane sürümleri bilgisayardakiyle aynı tutulur (sentence-transformers 6.0.1, transformers 5.12.1).

In [ ]:
if not os.path.exists("/content/rebulex"):
    !git clone -q -b {BRANCH} {REPO} /content/rebulex
else:
    !cd /content/rebulex && git fetch -q && git reset -q --hard origin/{BRANCH}
%cd /content/rebulex
!git log --oneline -1
!pip install -q -r requirements.txt sentence-transformers==6.0.1 transformers==5.12.1
import sentence_transformers, transformers, torch
print("sentence-transformers", sentence_transformers.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN yüklendi")
except Exception as e:
    print("HF_TOKEN yok; magibu-11b indirilemez, bu hücreyi tekrar çalıştır:", type(e).__name__)

## 4. Verileri hazırla
Zip Colab diskine açılır. Özet klasörü (`data/n2000/ozet`) doğrudan Drive'a bağlanır: her özet bittiği anda Drive'da olur.

In [ ]:
os.makedirs(SONUC, exist_ok=True)
if not os.path.exists(f"{LOCAL_DATA}/n2000/kararlar.parquet"):
    !unzip -q -o {DRIVE_DIR}/rerank_veri.zip -d {LOCAL_DATA}
if not os.path.exists(f"{LOCAL_DATA}/n2000/ozet"):
    os.symlink(SONUC, f"{LOCAL_DATA}/n2000/ozet")
!ls {LOCAL_DATA} {LOCAL_DATA}/n2000

## 5. Özetle + mini B araması
İlk çalıştırmada model (~22 GB) indirilir. Çıktı `ozet_sonuc/ozet.log`'a da yazılır. Bu hücre hata fırlatmaz,
böylece "Tümünü çalıştır" son hücreye (kapatma) mutlaka ulaşır.

In [ ]:
!python -u summarize_llm.py 2>&1 | tee -a {SONUC}/ozet.log

## 6. Özetleri göster

In [ ]:
import json, glob
for path in sorted(glob.glob(f"{SONUC}/T*.json")):
    s = json.load(open(path, encoding="utf-8"))
    print("=" * 100)
    print(f"{s['test_id']} | {s['kunye']}")
    print(f"karar {s['karar_kelime']} kelime, giriş {s['giris_token']} token -> özet {s['ozet_token']} token, "
          f"{s['ozet_kelime']} kelime, {s['sure_sn']} sn")
    print("=" * 100)
    print(s["ozet"])

## 7. Çalışma zamanını kapat
Sonuçlar Drive'da kalır (`REBULEX/ozet_sonuc/`).

In [ ]:
from google.colab import runtime
runtime.unassign()